# Experiment: Inspecting Nested Data

In [1]:

from datasets import load_dataset
import pandas as pd

# Load only the first 10 examples to save time
dataset = load_dataset("allenai/qasper", split="train[:10]", trust_remote_code=True)
sample = dataset[0]

# Exploring Scientific papers multiple sections
print(f"Paper Title: {sample['title']}")
print(f"Number of Sections: {len(sample['full_text']['section_name'])}")


/home/nguegnang/anaconda3/envs/qasper-rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Paper Title: Minimally Supervised Learning of Affective Events Using Discourse Relations
Number of Sections: 21


In [2]:
# Let's verify if paragraphs align with sections
sections = sample['full_text']['section_name']
paragraphs = sample['full_text']['paragraphs']

df = pd.DataFrame({
    "Section": sections,
    "Paragraph_Count": [len(p) for p in paragraphs],
    "First_Para_Preview": [p[0][:50] + "..." if p else "N/A" for p in paragraphs]
})

print(df.head(7))

                                             Section  Paragraph_Count  \
0                                       Introduction                4   
1                                       Related Work                5   
2                                    Proposed Method                1   
3              Proposed Method ::: Polarity Function                4   
4  Proposed Method ::: Discourse Relation-Based E...                3   
5  Proposed Method ::: Discourse Relation-Based E...                2   
6  Proposed Method ::: Discourse Relation-Based E...                2   

                                  First_Para_Preview  
0  Affective events BIBREF0 are events that typic...  
1  Learning affective events is closely related t...  
2                                                ...  
3                                                ...  
4  Our method requires a very small seed lexicon ...  
5  The seed lexicon matches (1) the latter event ...  
6  The seed lexicon matches ne

In [3]:
# Let's see the first actual text
if len(paragraphs[0]) > 0:
    print(f"Text Content: {paragraphs[0][0][:100]}...")

Text Content: Affective events BIBREF0 are events that typically affect people in positive or negative ways. For e...


In [15]:
print(type(dataset))   # <class 'datasets.arrow_dataset.Dataset'>


<class 'datasets.arrow_dataset.Dataset'>


In [16]:
print(dataset)          # Shows schema, num rows, and features


Dataset({
    features: ['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables'],
    num_rows: 10
})


In [17]:
print(dataset.features) # Shows column names and their types

{'id': Value(dtype='string', id=None), 'title': Value(dtype='string', id=None), 'abstract': Value(dtype='string', id=None), 'full_text': Sequence(feature={'section_name': Value(dtype='string', id=None), 'paragraphs': [Value(dtype='string', id=None)]}, length=-1, id=None), 'qas': Sequence(feature={'question': Value(dtype='string', id=None), 'question_id': Value(dtype='string', id=None), 'nlp_background': Value(dtype='string', id=None), 'topic_background': Value(dtype='string', id=None), 'paper_read': Value(dtype='string', id=None), 'search_query': Value(dtype='string', id=None), 'question_writer': Value(dtype='string', id=None), 'answers': Sequence(feature={'answer': {'unanswerable': Value(dtype='bool', id=None), 'extractive_spans': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'yes_no': Value(dtype='bool', id=None), 'free_form_answer': Value(dtype='string', id=None), 'evidence': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'highlighted_e

In [22]:
# Inspect Q&A structure across multiple samples
for i in range(min(3, len(dataset))):
    paper = dataset[i]
    qas = paper['qas']
    print(f"\n--- Paper {i}: {paper['title'][:80]} ---")
    print(f"  Sections: {len(paper['full_text']['section_name'])}")
    print(f"  Questions: {len(qas['question'])}")
    for j, q in enumerate(qas['question'][:3]):
        answers = qas['answers'][j]
        for a in answers['answer']:
            if a['unanswerable']:
                ans_text = "[unanswerable]"
            elif a['extractive_spans']:
                ans_text = " | ".join(a['extractive_spans'])
            elif a['free_form_answer']:
                ans_text = a['free_form_answer']
            elif a['yes_no'] is not None:
                ans_text = "Yes" if a['yes_no'] else "No"
            else:
                ans_text = "[no answer]"
        print(f"    Q{j+1}: {q[:80]}")
        print(f"        Answer: {ans_text[:100]}")


--- Paper 0: Minimally Supervised Learning of Affective Events Using Discourse Relations ---
  Sections: 21
  Questions: 9
    Q1: What is the seed lexicon?
        Answer: seed lexicon consists of positive and negative predicates
    Q2: What are the results?
        Answer: Using all data to train: AL -- BiGRU achieved 0.843 accuracy, AL -- BERT achieved 0.863 accuracy, AL
    Q3: How are relations used to propagate polarity?
        Answer: cause relation: both events in the relation should have the same polarity; concession relation: even

--- Paper 1: PO-EMO: Conceptualization, Annotation, and Modeling of Aesthetic Emotions in Ger ---
  Sections: 25
  Questions: 3
    Q1: Does the paper report macro F1?
        Answer: Yes
    Q2: How is the annotation experiment evaluated?
        Answer: confusion matrices of labels between annotators
    Q3: What are the aesthetic emotions formalized?
        Answer: feelings of suspense experienced in narratives not only respond to the trajec

In [19]:
# Summary statistics across all 10 samples
stats = []
for i in range(len(dataset)):
    paper = dataset[i]
    num_sections = len(paper['full_text']['section_name'])
    num_paragraphs = sum(len(p) for p in paper['full_text']['paragraphs'])
    num_questions = len(paper['qas']['question'])
    total_answers = sum(len(a['answer']) for a in paper['qas']['answers'])
    stats.append({
        "Paper": paper['title'][:50] + "...",
        "Sections": num_sections,
        "Paragraphs": num_paragraphs,
        "Questions": num_questions,
        "Total_Answers": total_answers,
    })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))
print(f"\n--- Averages ---")
print(f"  Sections/paper:   {stats_df['Sections'].mean():.1f}")
print(f"  Paragraphs/paper: {stats_df['Paragraphs'].mean():.1f}")
print(f"  Questions/paper:  {stats_df['Questions'].mean():.1f}")

                                                Paper  Sections  Paragraphs  Questions  Total_Answers
Minimally Supervised Learning of Affective Events ...        21          72          9             12
PO-EMO: Conceptualization, Annotation, and Modelin...        25          80          3              4
Community Identity and User Engagement in a Multi-...        14          78          6              8
Question Answering based Clinical Text Structuring...        19          47         12             16
  Progress and Tradeoffs in Neural Language Models...         7          26          3              4
Stay On-Topic: Generating Context-specific Fake Re...         5         644          6              8
Saliency Maps Generation for Automatic Text Summar...        12          34          3              4
  Probabilistic Bias Mitigation in Word Embeddings...        14          52          3              4
Massive vs. Curated Word Embeddings for Low-Resour...        15          58       

In [20]:
# Deep dive into answer types and evidence
from collections import Counter

answer_types = []
evidence_counts = []
for i in range(len(dataset)):
    for ans_list in dataset[i]['qas']['answers']:
        for ans in ans_list['answer']:
            if ans['unanswerable']:
                answer_types.append('unanswerable')
            elif ans['extractive_spans']:
                answer_types.append('extractive')
            elif ans['free_form_answer']:
                answer_types.append('free_form')
            elif ans['yes_no'] is not None:
                answer_types.append('yes_no')
            else:
                answer_types.append('other')
            evidence_counts.append(len(ans['evidence']))

type_counts = Counter(answer_types)
print("Answer Type Distribution:")
for atype, count in type_counts.most_common():
    print(f"  {atype:15s}: {count:3d} ({count/len(answer_types)*100:.1f}%)")
print(f"\nAvg evidence paragraphs per answer: {sum(evidence_counts)/len(evidence_counts):.2f}")

Answer Type Distribution:
  extractive     :  32 (47.1%)
  free_form      :  18 (26.5%)
  unanswerable   :  11 (16.2%)
  yes_no         :   7 (10.3%)

Avg evidence paragraphs per answer: 1.31


## Characters vs. Tokens

In [5]:
from transformers import AutoTokenizer

# We use the specific tokenizer for our embedding model (BGE)
# If we used a GPT tokenizer, the counts would be wrong!
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

text = "RAG is a technique that combines retrieval and generation."
chars = len(text)
tokens = len(tokenizer.encode(text))

print(f"Character count: {chars}")
print(f"Token count: {tokens}")
print(f"Ratio: {chars / tokens:.2f} chars per token (approx)")

Character count: 58
Token count: 12
Ratio: 4.83 chars per token (approx)


## The Sliding Window Effect

In [8]:
# Create a realistic text where each word is unique and distinguishable
long_text = (
    "Retrieval augmented generation combines information retrieval with language models "
    "to produce grounded answers from external knowledge sources. This technique first "
    "retrieves relevant documents from a corpus and then feeds them as context to the "
    "generator model. The result is more factual and verifiable output compared to pure "
    "generation approaches that rely solely on parametric memory stored during training."
)

def simulate_window(text, max_len=20, overlap=5):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    stride = max_len - overlap  # The step size

    print(f"Total Tokens: {len(tokens)}")
    print(f"Window Size: {max_len}, Overlap: {overlap}, Stride: {stride}")

    chunks = []
    for i in range(0, len(tokens), stride):
        chunk_tokens = tokens[i : i + max_len]
        chunks.append(tokenizer.decode(chunk_tokens))
    return chunks

chunks = simulate_window(long_text)
print("-" * 60)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: \"{chunk}\"")

# Highlight the overlap between consecutive chunks
max_len = 20
overlap = 5
print("\n" + "=" * 60)
print("OVERLAP VERIFICATION:")
for i in range(len(chunks) - 1):
    words_c1 = chunks[i].split()
    words_c2 = chunks[i + 1].split()
    tail = " ".join(words_c1[-overlap:])
    head = " ".join(words_c2[:overlap])
    print(f"\n  Chunk {i+1} tail:  \"{tail}\"")
    print(f"  Chunk {i+2} head:  \"{head}\"")
    print(f"  Match: {tail == head}")

Total Tokens: 66
Window Size: 20, Overlap: 5, Stride: 15
------------------------------------------------------------
Chunk 1: "retrieval augmented generation combines information retrieval with language models to produce grounded answers from external knowledge sources. this technique"
Chunk 2: "knowledge sources. this technique first retrieves relevant documents from a corpus and then feeds them as context to"
Chunk 3: "feeds them as context to the generator model. the result is more factual and verifiable output"
Chunk 4: "verifiable output compared to pure generation approaches that rely solely on parametric memory stored during training"
Chunk 5: "##metric memory stored during training."

OVERLAP VERIFICATION:

  Chunk 1 tail:  "external knowledge sources. this technique"
  Chunk 2 head:  "knowledge sources. this technique first"
  Match: False

  Chunk 2 tail:  "feeds them as context to"
  Chunk 3 head:  "feeds them as context to"
  Match: True

  Chunk 3 tail:  "more factual and

In [9]:
# Token-level overlap verification
for i in range(len(chunks) - 1):
    tokens_c1 = tokenizer.encode(chunks[i], add_special_tokens=False)
    tokens_c2 = tokenizer.encode(chunks[i + 1], add_special_tokens=False)
    tail = tokens_c1[-overlap:]
    head = tokens_c2[:overlap]
    print(f"\n  Chunk {i+1} tail tokens: {tail}")
    print(f"  Chunk {i+2} head tokens: {head}")
    print(f"  Match: {tail == head}")


  Chunk 1 tail tokens: [3716, 4216, 1012, 2023, 6028]
  Chunk 2 head tokens: [3716, 4216, 1012, 2023, 6028]
  Match: True

  Chunk 2 tail tokens: [14172, 2068, 2004, 6123, 2000]
  Chunk 3 head tokens: [14172, 2068, 2004, 6123, 2000]
  Match: True

  Chunk 3 tail tokens: [2310, 3089, 22749, 3468, 6434]
  Chunk 4 head tokens: [2310, 3089, 22749, 3468, 6434]
  Match: True

  Chunk 4 tail tokens: [12589, 3638, 8250, 2076, 2731]
  Chunk 5 head tokens: [1001, 1001, 12046, 3638, 8250]
  Match: False


## Chunking QASPER

In [10]:
from transformers import AutoTokenizer
from typing import List, Dict

class QasperChunker:
    def __init__(self, model_name="BAAI/bge-small-en-v1.5", max_tokens=512, overlap_pct=0.1):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_tokens = max_tokens
        # Calculate overlap tokens (e.g., 512 * 0.1 = ~51 tokens)
        self.overlap_tokens = int(max_tokens * overlap_pct)

    def split_large_text(self, text: str) -> List[str]:
        """
        Splits a single long text string into overlapping chunks based on token count.
        """
        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        
        # Case 1: Text fits in one chunk
        if len(tokens) <= self.max_tokens:
            return [text]
        
        # Case 2: Sliding Window
        chunks = []
        stride = self.max_tokens - self.overlap_tokens
        
        # range(start, stop, step) -> We step by 'stride', not 'max_tokens'
        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i : i + self.max_tokens]
            chunk_text = self.tokenizer.decode(chunk_tokens)
            chunks.append(chunk_text)
            
        return chunks

    def process_paper(self, paper_data: Dict) -> List[Dict]:
        """
        Flattens a paper into a list of chunk dictionaries with metadata.
        """
        chunks = []
        paper_id = paper_data['id']
        title = paper_data['title']
        
        # Iterate over sections
        sections = paper_data['full_text']['section_name']
        paragraphs_list = paper_data['full_text']['paragraphs']

        for section_idx, (section_name, section_paras) in enumerate(zip(sections, paragraphs_list)):
            # Iterate over paragraphs in that section
            for para_idx, paragraph in enumerate(section_paras):
                
                # Base Metadata (Crucial for Citation)
                base_meta = {
                    "paper_id": paper_id,
                    "title": title,
                    "section_name": section_name,
                    "original_para_id": f"{paper_id}_{section_idx}_{para_idx}"
                }

                # Split paragraph if needed
                text_chunks = self.split_large_text(paragraph)
                
                for sub_idx, text in enumerate(text_chunks):
                    # Create the final chunk object
                    chunk = base_meta.copy()
                    chunk["chunk_id"] = f"{base_meta['original_para_id']}_{sub_idx}"
                    chunk["text"] = text
                    chunks.append(chunk)
                    
        return chunks

In [11]:
# Test QasperChunker on the first paper
chunker = QasperChunker(max_tokens=512, overlap_pct=0.1)
paper = dataset[0]

chunks = chunker.process_paper(paper)

print(f"Paper: {paper['title'][:80]}")
print(f"Total chunks produced: {len(chunks)}")
print("=" * 60)

# Show first 3 chunks
for i, chunk in enumerate(chunks[:3]):
    token_count = len(chunker.tokenizer.encode(chunk["text"], add_special_tokens=False))
    print(f"\nChunk {i+1}:")
    print(f"  ID:      {chunk['chunk_id']}")
    print(f"  Section: {chunk['section_name']}")
    print(f"  Tokens:  {token_count}")
    print(f"  Text:    {chunk['text'][:120]}...")

# Summary: how many chunks per section
print("\n" + "=" * 60)
print("Chunks per section:")
from collections import Counter
section_counts = Counter(c["section_name"] for c in chunks)
for section, count in section_counts.items():
    print(f"  {section[:50]:50s} → {count} chunk(s)")

# Check if any chunk exceeds the token limit
max_chunk_tokens = max(
    len(chunker.tokenizer.encode(c["text"], add_special_tokens=False)) for c in chunks
)
print(f"\nMax tokens in any chunk: {max_chunk_tokens} (limit: {chunker.max_tokens})")

Paper: Minimally Supervised Learning of Affective Events Using Discourse Relations
Total chunks produced: 72

Chunk 1:
  ID:      1909.00694_0_0_0
  Section: Introduction
  Tokens:  125
  Text:    Affective events BIBREF0 are events that typically affect people in positive or negative ways. For example, getting mone...

Chunk 2:
  ID:      1909.00694_0_1_0
  Section: Introduction
  Tokens:  75
  Text:    Learning affective events is challenging because, as the examples above suggest, the polarity of an event is not necessa...

Chunk 3:
  ID:      1909.00694_0_2_0
  Section: Introduction
  Tokens:  342
  Text:    In this paper, we propose a simple and effective method for learning affective events that only requires a very small se...

Chunks per section:
  Introduction                                       → 4 chunk(s)
  Related Work                                       → 5 chunk(s)
  Proposed Method                                    → 1 chunk(s)
  Proposed Method ::: Polarity Functi

## Vector Similarity

In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# 1. Define sentences with semantic relationships
sentences = [
    "The patient has a fracture in the tibia.",   # 0: Anchor
    "The subject suffered a broken leg bone.",    # 1: Semantically identical (synonyms)
    "The chef cooked a delicious meal.",          # 2: Completely unrelated
    "The patient has a fracture in the humerus."  # 3: Syntactically similar, medically different location
]

# 2. Encode WITH normalization
# normalize_embeddings=True is MANDATORY for Inner Product to work as Cosine Similarity
embeddings = model.encode(sentences, normalize_embeddings=True)

# 3. Calculate Dot Product between Anchor (0) and others
# We use numpy dot product
scores = np.dot(embeddings[0], embeddings.T)

print(f"Anchor: {sentences[0]}")
print(f"1. Synonym Match Score: {scores[1]:.4f}") 
print(f"2. Unrelated Score:     {scores[2]:.4f}")
print(f"3. Similar Word Score:  {scores[3]:.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 653.64it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Anchor: The patient has a fracture in the tibia.
1. Synonym Match Score: 0.8380
2. Unrelated Score:     0.5014
3. Similar Word Score:  0.8990


In [14]:
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from typing import List, Dict

class DenseIndexer:
    def __init__(self, model_name="BAAI/bge-small-en-v1.5", index_path="data/indices/dense.index"):
        self.model = SentenceTransformer(model_name)
        self.index_path = index_path
        self.dimension = 384 # Specific to BGE-Small
        self.index = None
        self.metadata_store = [] # FAISS only stores numbers; we need to store the text separately

    def build_index(self, chunks: List[Dict]):
        print(f"Generating embeddings for {len(chunks)} chunks...")
        
        texts = [c['text'] for c in chunks]
        
        # 1. Encode
        # normalize_embeddings=True prepares vectors for Inner Product search
        embeddings = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
        
        # 2. Initialize FAISS Index
        # IndexFlatIP = Exact search using Inner Product (fastest for small datasets)
        self.index = faiss.IndexFlatIP(self.dimension)
        
        # 3. Add vectors (must be float32 for FAISS)
        self.index.add(np.array(embeddings).astype('float32'))
        
        # 4. Store metadata
        self.metadata_store = chunks
        print(f"Dense Index built with {self.index.ntotal} vectors.")

    def save(self):
        # Save the FAISS index (the vectors)
        faiss.write_index(self.index, self.index_path)
        
        # Save the metadata (the text and IDs) using pickle
        with open(self.index_path + ".meta", "wb") as f:
            pickle.dump(self.metadata_store, f)
        print(f"Saved dense index to {self.index_path}")

In [19]:
# Test DenseIndexer: chunk → embed → index → save
# 1. Chunk all 10 papers
chunker = QasperChunker(max_tokens=512, overlap_pct=0.1)
all_chunks = []
for i in range(len(dataset)):
    all_chunks.extend(chunker.process_paper(dataset[i]))
print(f"Total chunks across all papers: {len(all_chunks)}")

# 2. Build the dense index
#indexer = DenseIndexer(index_path="data/indices/dense.index")
indexer = DenseIndexer(index_path="../data/indices/dense.index")
indexer.build_index(all_chunks)

# 3. Save to disk
indexer.save()

# 4. Quick search test — query the index directly
query = "What methods are used for text classification?"
query_embedding = indexer.model.encode([query], normalize_embeddings=True).astype('float32')

k = 4  # top-4 results
scores, ids = indexer.index.search(query_embedding, k)

print(f"\nQuery: \"{query}\"")
print(f"Top-{k} results:")
for rank, (score, idx) in enumerate(zip(scores[0], ids[0])):
    chunk = indexer.metadata_store[idx]
    print(f"\n  #{rank+1} (score: {score:.4f})")
    print(f"    Paper:   {chunk['title'][:60]}")
    print(f"    Section: {chunk['section_name']}")
    print(f"    Text:    {chunk['text'][:120]}...")

Token indices sequence length is longer than the specified maximum sequence length for this model (888 > 512). Running this sequence through the model will result in indexing errors


Total chunks across all papers: 1116


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1169.29it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for 1116 chunks...


Batches: 100%|██████████| 35/35 [00:34<00:00,  1.00it/s]

Dense Index built with 1116 vectors.
Saved dense index to ../data/indices/dense.index

Query: "What methods are used for text classification?"
Top-4 results:

  #1 (score: 0.7313)
    Paper:   Question Answering based Clinical Text Structuring Using Pre
    Section: The Proposed Model for QA-CTS Task
    Text:    In this section, we present an effective model for the question answering based clinical text structuring (QA-CTS). As s...

  #2 (score: 0.7268)
    Paper:   Question Answering based Clinical Text Structuring Using Pre
    Section: Conclusion
    Text:    In this paper, we present a question answering based clinical text structuring (QA-CTS) task, which unifies different cl...

  #3 (score: 0.7257)
    Paper:   Is there Gender bias and stereotype in Portuguese Word Embed
    Section: Related Work
    Text:    There is a wide range of techniques that provide interesting results in the context of ML algorithms geared to the class...

  #4 (score: 0.7182)
    Paper:   Question A

## The Keyword Space (Sparse Index)


In [22]:
# CELL 5
from nltk.tokenize import word_tokenize
import nltk

# Download NLTK data (only needed once)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

query = "Error-Rate: 404"

# 1. Simple Split (Bad)
print(f"Simple Split: {query.split()}")
# Output: ['Error-Rate:', '404'] -> Punctuation is stuck to the word!

# 2. NLTK Tokenization (Good)
print(f"NLTK: {word_tokenize(query)}")
# Output: ['Error-Rate', ':', '404'] -> Punctuation is separated.

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/nguegnang/nltk_data...


Simple Split: ['Error-Rate:', '404']
NLTK: ['Error-Rate', ':', '404']


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [23]:
from rank_bm25 import BM25Okapi
import pickle
import nltk
from nltk.tokenize import word_tokenize
from typing import List, Dict

# Ensure NLTK data is present
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

class SparseIndexer:
    def __init__(self, index_path="data/indices/sparse.pkl"):
        self.index_path = index_path
        self.bm25 = None
        self.corpus_chunks = []

    def build_index(self, chunks: List[Dict]):
        print("Tokenizing corpus for BM25...")
        # 1. Tokenize: Break text into list of words ["the", "cat", ...]
        # We use .lower() to make it case-insensitive
        tokenized_corpus = [word_tokenize(doc['text'].lower()) for doc in chunks]
        
        # 2. Build Index: BM25Okapi calculates IDF for every word immediately
        self.bm25 = BM25Okapi(tokenized_corpus)
        self.corpus_chunks = chunks
        print("BM25 Index built.")

    def save(self):
        # BM25 is a Python object, so we pickle it directly
        with open(self.index_path, "wb") as f:
            pickle.dump({
                "model": self.bm25,
                "metadata": self.corpus_chunks
            }, f)
        print(f"Saved sparse index to {self.index_path}")

In [25]:
# Test SparseIndexer: chunk → tokenize → index → save → search
# 1. Use the same chunks as for DenseIndexer (all_chunks)

sparse_indexer = SparseIndexer(index_path="../data/indices/sparse.pkl")
sparse_indexer.build_index(all_chunks)

# 2. Save to disk
sparse_indexer.save()

# 3. Quick search test — query the sparse index directly
query = "What methods are used for text classification?"
query_tokens = word_tokenize(query.lower())

k = 4  # top-4 results
scores = sparse_indexer.bm25.get_scores(query_tokens)
top_k_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]

print(f"\nQuery: \"{query}\"")
print(f"Top-{k} results:")
for rank, idx in enumerate(top_k_idx):
    chunk = sparse_indexer.corpus_chunks[idx]
    print(f"\n  #{rank+1} (score: {scores[idx]:.4f})")
    print(f"    Paper:   {chunk['title'][:60]}")
    print(f"    Section: {chunk['section_name']}")
    print(f"    Text:    {chunk['text'][:120]}...")


Tokenizing corpus for BM25...
BM25 Index built.
Saved sparse index to ../data/indices/sparse.pkl

Query: "What methods are used for text classification?"
Top-4 results:

  #1 (score: 13.7844)
    Paper:   Stay On-Topic: Generating Context-specific Fake Restaurant R
    Section: Generative Model
    Text:    \newblock What yelp fake review filter might be doing?...

  #2 (score: 11.5181)
    Paper:   Question Answering based Clinical Text Structuring Using Pre
    Section: Related Work ::: Clinical Text Structuring
    Text:    Clinical text structuring is a final problem which is highly related to practical applications. Most of existing studies...

  #3 (score: 10.2352)
    Paper:   Stay On-Topic: Generating Context-specific Fake Restaurant R
    Section: Generative Model
    Text:    \newblock The anatomy of online deception: What makes automated text...

  #4 (score: 10.0640)
    Paper:   Question Answering based Clinical Text Structuring Using Pre
    Section: Experimental Studies 